# 77. The THEMIS Mission — Implementation / THEMIS 미션 구현

**Paper**: Angelopoulos, V. "The THEMIS Mission", *Space Science Reviews* **141**, 5–34, 2008. DOI: 10.1007/s11214-008-9336-1

## Scope / 범위

This notebook reproduces, with synthetic data and analytic models, four hallmark aspects of the THEMIS mission design and science:
1. The five-probe apogee-aligned constellation orbits (P1/P2/P3/P4/P5 at ~30/19/12/12/10 R_E apogee)
2. Substorm trigger timing comparison between the Near-Earth Neutral Line (NENL) and Current Disruption (CD) paradigms
3. Magnetotail multi-point reconnection signatures (B_z reversal, dispersed flow bursts)
4. Ground-Based Observatory (GBO) conjugate mapping from the magnetotail to the auroral ionosphere

이 노트북은 합성 데이터와 분석 모델을 사용하여 THEMIS 미션 설계와 과학의 네 가지 대표적 측면을 재현한다:
1. 5위성 apogee 정렬 군집 궤도 (P1/P2/P3/P4/P5 약 30/19/12/12/10 R_E apogee)
2. NENL와 CD 두 패러다임 사이의 부폭풍 트리거 timing 비교
3. Magnetotail 다지점 reconnection 시그너처 (B_z 역전, 분산된 flow burst)
4. Ground-Based Observatory(GBO) magnetotail-to-ionosphere 공역 사상

All data are synthetic; the goal is to make the geometric, temporal, and physical arguments of the paper executable.

모든 데이터는 합성이며, 목표는 논문의 기하학적·시간적·물리적 논증을 실행 가능한 형태로 만드는 것이다.

In [ ]:
"""Imports and global configuration.

We use NumPy for vectorized math, SciPy for orbital mechanics helpers,
and Matplotlib for plotting. Earth radius (R_E) is the canonical length unit.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge
from scipy.optimize import brentq

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants in SI; only used where needed.
R_E_KM = 6371.0       # Earth radius (km)
MU_E = 398600.4418    # Earth GM (km^3/s^2)
SIDEREAL_DAY_S = 86164.0905  # sidereal day (s)

RNG = np.random.default_rng(seed=20080202)  # 1st tail-season epoch as seed

## Part 1: Five-Probe Apogee-Aligned Constellation / 5위성 Apogee 정렬 군집

THEMIS placed five probes on highly elliptical orbits with **integer-ratio periods** (4 : 2 : 1 : 1 : 7/8 sidereal days for P1 : P2 : P3 : P4 : P5) so that all five probes reach apogee on the **same midnight meridian** repeatedly. The apogee distances are 30 / 19 / 12 / 12 / 10 R_E, with P3 and P4 nearly co-located but offset by ~1 R_E in mean anomaly. We compute Keplerian orbits in the ecliptic-projected plane and verify the conjunction geometry.

THEMIS는 5위성을 정수배 주기 (P1:P2:P3:P4:P5 = 4:2:1:1:7/8 sidereal days)로 배치하여 다섯 위성이 같은 자정 자오선의 apogee에 반복적으로 정렬되도록 했다. Apogee 거리는 각각 30/19/12/12/10 R_E이며, P3와 P4는 같은 apogee를 갖되 mean anomaly로 ~1 R_E 떨어져 있다. 여기서는 ecliptic 투영면에서 Kepler 궤도를 계산하여 conjunction 기하를 확인한다.

In [ ]:
def solve_kepler(mean_anomaly: np.ndarray, eccentricity: float, tol: float = 1e-10) -> np.ndarray:
    """Solve Kepler's equation M = E - e*sin(E) for the eccentric anomaly E.

    Args:
        mean_anomaly: Mean anomaly array (radians).
        eccentricity: Orbital eccentricity (0 <= e < 1).
        tol: Convergence tolerance on |E - e*sin(E) - M|.

    Returns:
        Eccentric anomaly E (radians), same shape as ``mean_anomaly``.
    """
    E = mean_anomaly.copy()
    for _ in range(50):
        delta = (E - eccentricity * np.sin(E) - mean_anomaly) / (1.0 - eccentricity * np.cos(E))
        E -= delta
        if np.max(np.abs(delta)) < tol:
            break
    return E


def kepler_orbit_xy(
    apogee_re: float,
    perigee_km: float,
    period_days: float,
    raan_deg: float,
    aper_deg: float,
    n_samples: int = 720,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate the X-Y trace of a Keplerian orbit projected on the equatorial plane.

    Args:
        apogee_re: Apogee distance in Earth radii.
        perigee_km: Perigee altitude above Earth's surface (km).
        period_days: Orbital period (days). Used only for time array, not for positions.
        raan_deg: Right Ascension of Ascending Node (degrees).
        aper_deg: Argument of perigee (degrees).
        n_samples: Number of true-anomaly samples.

    Returns:
        Tuple of (x_re, y_re, time_hr) where positions are in R_E and time is in hours.
    """
    r_apo = apogee_re * R_E_KM
    r_peri = R_E_KM + perigee_km
    a = 0.5 * (r_apo + r_peri)
    e = (r_apo - r_peri) / (r_apo + r_peri)
    period_s = period_days * SIDEREAL_DAY_S
    t = np.linspace(0.0, period_s, n_samples)
    M = 2.0 * np.pi * t / period_s
    E = solve_kepler(M, e)
    nu = 2.0 * np.arctan2(np.sqrt(1.0 + e) * np.sin(0.5 * E), np.sqrt(1.0 - e) * np.cos(0.5 * E))
    r = a * (1.0 - e * np.cos(E))
    rap = np.deg2rad(raan_deg + aper_deg)
    x = (r * np.cos(nu + rap)) / R_E_KM
    y = (r * np.sin(nu + rap)) / R_E_KM
    return x, y, t / 3600.0


# Probe parameters at the 1st tail-season epoch (Table 5 of the paper, simplified).
# In GSM the magnetotail extends to negative X, so we orient line of apsides toward -X.
PROBES = {
    'P1 (TH-B)':  {'apogee_re': 30.0, 'period_days': 4.000,    'color': '#d62728'},
    'P2 (TH-C)':  {'apogee_re': 19.0, 'period_days': 2.000,    'color': '#ff7f0e'},
    'P3 (TH-D)':  {'apogee_re': 12.0, 'period_days': 1.000,    'color': '#2ca02c'},
    'P4 (TH-E)':  {'apogee_re': 12.0, 'period_days': 1.000,    'color': '#1f77b4'},
    'P5 (TH-A)':  {'apogee_re': 10.0, 'period_days': 7.0/8.0,  'color': '#9467bd'},
}

# Common perigee altitude and tail-aligned line of apsides.
PERIGEE_KM = 437.0  # km (paper Sect. 3.1)
RAP_DEG = 180.0     # line of apsides points to -X (anti-sunward)

fig, ax = plt.subplots(figsize=(10, 8))
for name, p in PROBES.items():
    x, y, _ = kepler_orbit_xy(p['apogee_re'], PERIGEE_KM, p['period_days'], RAP_DEG, 0.0)
    ax.plot(x, y, color=p['color'], lw=1.6, label=f"{name}: R_A = {p['apogee_re']:.0f} R_E, T = {p['period_days']:.3f} d")
    ax.plot(-p['apogee_re'], 0.0, 'o', color=p['color'], ms=10, mec='k', mew=0.8)
ax.add_patch(Circle((0, 0), 1.0, color='deepskyblue', alpha=0.7, zorder=5))
ax.add_patch(Wedge((0, 0), 1.0, 90, 270, color='gray', alpha=0.7, zorder=6))
ax.axhline(0, color='k', lw=0.4)
ax.axvline(0, color='k', lw=0.4)
ax.set_xlim(-35, 5)
ax.set_ylim(-12, 12)
ax.set_aspect('equal')
ax.set_xlabel('X_GSM (R_E)')
ax.set_ylabel('Y_GSM (R_E)')
ax.set_title('THEMIS 1st Tail Season Orbits — Apogees on Midnight Meridian / 자정 자오선 정렬 궤도')
ax.legend(loc='lower left', fontsize=9)
ax.text(-30, 11, 'Solar wind →', fontsize=10, color='goldenrod')
plt.tight_layout()
plt.show()

### 1.1 Conjunction Recurrence / Conjunction 반복

Because every probe period is an integer or simple-rational multiple of one sidereal day, the apogee-aligned conjunction repeats on a regular cadence. We sample the probes' radial distances over a 16-day window and identify times when all five are within ±2 R_E of their apogee on the midnight meridian.

각 위성 주기가 sidereal day의 정수 또는 간단한 유리수 배수이므로, apogee 정렬 conjunction은 일정 주기로 반복된다. 16일 동안의 위성 거리를 샘플링하여 5위성이 모두 자정 자오선의 apogee에서 ±2 R_E 이내에 있는 시간을 찾아낸다.

In [ ]:
def radial_distance(apogee_re: float, perigee_km: float, period_days: float, t_days: np.ndarray) -> np.ndarray:
    """Compute geocentric distance (R_E) at given times for a Kepler orbit.

    Args:
        apogee_re: Apogee in R_E.
        perigee_km: Perigee altitude in km.
        period_days: Orbital period in days.
        t_days: Time array in days.

    Returns:
        Geocentric distance array in R_E.
    """
    r_apo = apogee_re * R_E_KM
    r_peri = R_E_KM + perigee_km
    a = 0.5 * (r_apo + r_peri)
    e = (r_apo - r_peri) / (r_apo + r_peri)
    period_s = period_days * SIDEREAL_DAY_S
    M = 2.0 * np.pi * (t_days * 86400.0) / period_s
    E = solve_kepler(M, e)
    return (a * (1.0 - e * np.cos(E))) / R_E_KM


t_days = np.linspace(0.0, 16.0, 16 * 96)  # 15-min cadence
fig, ax = plt.subplots(figsize=(11, 5))
for name, p in PROBES.items():
    r = radial_distance(p['apogee_re'], PERIGEE_KM, p['period_days'], t_days)
    ax.plot(t_days, r, color=p['color'], lw=1.4, label=name)
    ax.axhline(p['apogee_re'], color=p['color'], lw=0.4, ls=':', alpha=0.5)

# Mark conjunctions: when P1 is within 1 R_E of its apogee (every 4 days).
for d in (0, 4, 8, 12, 16):
    ax.axvline(d, color='k', lw=0.6, ls='--', alpha=0.5)
    ax.text(d, 31, f'major\nT+{d}d', ha='center', fontsize=8)

ax.set_xlabel('Time since 1st-tail epoch (days)')
ax.set_ylabel('Geocentric distance (R_E)')
ax.set_title('Probe radial distances: P1 4d, P2 2d, P3=P4 1d, P5 7/8 d / 위성 거리')
ax.legend(ncol=5, fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.18))
ax.set_ylim(0, 33)
plt.tight_layout()
plt.show()

# Quantify the recurrence.
print("Period ratios (should match 4 : 2 : 1 : 1 : 7/8):")
for name, p in PROBES.items():
    print(f"  {name}: T = {p['period_days']:.4f} sidereal days")
print("\nMajor conjunctions (all 5 probes near apogee) recur every 4 sidereal days.")
print("Minor conjunctions (P2 + inner) recur every 2 sidereal days.")

## Part 2: Substorm Trigger — NENL vs CD Timing / 부폭풍 트리거 모델 비교

The two competing paradigms predict opposite chronologies (Tables 2 and 3 of the paper):
- **CD model**: current disruption at 8–10 R_E at t = 0 → auroral break-up at t = 30 s → reconnection at ~25 R_E at t = 60 s. A fast rarefaction wave (V_x ~ −1600 km/s) propagates tailward.
- **NENL model**: reconnection at ~25 R_E at t = 0 → bursty bulk flows (BBFs) reach the inner plasma sheet at t = 90 s → current disruption at t = 90 s → break-up at t = 120 s.

We synthesize the times of arrival at probes P5 (~10 R_E), P3/P4 (~12 R_E), P2 (~19 R_E), and P1 (~30 R_E) under each model, propagating with the model-specified wave/flow speeds.

두 패러다임의 chronological prediction은 정반대다. CD 모델은 t=0이 8–10 R_E이고 빠른 rarefaction wave가 tailward로 V_x ~ -1600 km/s로 전파, NENL 모델은 t=0이 ~25 R_E이고 BBF가 earthward로 흐른다. 각 모델 하에서 P5, P3/P4, P2, P1에서의 신호 도달 시간을 합성한다.

In [ ]:
def cd_arrival_time(x_probe_re: float, x_source_re: float = -9.0, v_rarefaction_km_s: float = -1600.0) -> float:
    """Compute when a tailward rarefaction wave from the CD source reaches a probe.

    The wave starts at x_source_re at t=0 and travels with negative (tailward) velocity
    in the GSM X-direction. Probes earthward of the source (x_probe > x_source) are
    not reached and return NaN.

    Args:
        x_probe_re: Probe X position in R_E (negative = tailward).
        x_source_re: Source location of CD onset (R_E, paper uses 8-10 R_E so -9).
        v_rarefaction_km_s: Wave velocity in km/s (negative = tailward).

    Returns:
        Arrival time at the probe (s); NaN if probe is upstream of source.
    """
    if x_probe_re > x_source_re:
        return np.nan
    distance_km = (x_source_re - x_probe_re) * R_E_KM
    return distance_km / abs(v_rarefaction_km_s)


def nenl_arrival_time(x_probe_re: float, x_source_re: float = -25.0, v_bbf_km_s: float = 800.0) -> float:
    """Compute when a BBF launched at the NENL reaches a probe earthward.

    Probes tailward of the source (x_probe < x_source) are reached by the
    tailward-moving plasmoid (~600 km/s). We model both branches.

    Args:
        x_probe_re: Probe X position in R_E (negative = tailward).
        x_source_re: NENL X position (R_E).
        v_bbf_km_s: BBF earthward speed in km/s (positive).

    Returns:
        Arrival time at the probe (s).
    """
    if x_probe_re >= x_source_re:
        # Earthward branch — BBF (V_x = +800 km/s).
        distance_km = (x_probe_re - x_source_re) * R_E_KM
        return distance_km / v_bbf_km_s
    # Tailward branch — plasmoid at ~600 km/s.
    distance_km = (x_source_re - x_probe_re) * R_E_KM
    return distance_km / 600.0


# Probe X positions at apogee (negative because the magnetotail is in -X_GSM).
PROBE_X = {
    'P5':  -10.0,
    'P3':  -12.0,
    'P4':  -12.0,
    'P2':  -19.0,
    'P1':  -30.0,
}

# Auroral onset offsets (CD: +30 s, NENL: +120 s).
AURORAL_OFFSETS = {'CD': 30.0, 'NENL': 120.0}

results = []
for probe, xp in PROBE_X.items():
    cd_t = cd_arrival_time(xp)
    nenl_t = nenl_arrival_time(xp)
    results.append((probe, xp, cd_t, nenl_t, nenl_t - cd_t))

print(f"{'Probe':<6} {'X (R_E)':>9} {'CD t (s)':>10} {'NENL t (s)':>12} {'Δt (s)':>10}")
print('-' * 50)
for probe, xp, cdt, nt, dt in results:
    print(f"{probe:<6} {xp:9.1f} {cdt:10.2f} {nt:12.2f} {dt:10.2f}")

print('\nAuroral break-up time (ground): CD = +30 s after CD source onset')
print('                                NENL = +120 s after NENL source onset')

In [ ]:
# Visualize the two timing scenarios on a stack plot.
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

probe_order = ['P5', 'P3', 'P4', 'P2', 'P1']
y_pos = np.arange(len(probe_order))

for ax, model in zip(axes, ('CD', 'NENL')):
    arrivals = []
    for probe in probe_order:
        xp = PROBE_X[probe]
        t = cd_arrival_time(xp) if model == 'CD' else nenl_arrival_time(xp)
        arrivals.append(t)
    arrivals = np.array(arrivals)

    ax.barh(y_pos, arrivals, color='steelblue', edgecolor='k', alpha=0.7)
    for y, t, probe in zip(y_pos, arrivals, probe_order):
        ax.text(t + 2, y, f'{t:.1f} s', va='center', fontsize=10)
    ax.axvline(AURORAL_OFFSETS[model], color='red', lw=2, ls='--', label=f'auroral break-up ({model}): t = {AURORAL_OFFSETS[model]:.0f} s')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(probe_order)
    ax.set_xlabel('Time after source onset (s)')
    ax.set_title(f'{model} model — signal arrival at each probe / 신호 도달 시간')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 200)

plt.tight_layout()
plt.show()

print('\nDecision rule (paper Sect. 2.1):')
print('  If signal reaches inner probes (P5/P3/P4) BEFORE outer (P2/P1): CD model.')
print('  If signal reaches outer probes (P1/P2) BEFORE inner (P5/P3/P4): NENL model.')
print('  THEMIS resolved this in 2008 (Angelopoulos et al., Science 321, 931): NENL precedes by ~1.5 min.')

## Part 3: Multi-Point Reconnection Signatures / 다지점 Reconnection 시그너처

Magnetic reconnection in the magnetotail produces canonical signatures:
- **B_z reversal**: probes earthward of the X-line see B_z > 0 (dipolarization), tailward see B_z < 0 (plasmoid).
- **Plasma jets**: V_x > 0 earthward of the X-line (BBFs ~800 km/s), V_x < 0 tailward (plasmoid ~600 km/s).
- **Energetic particle dispersion**: faster particles arrive earlier.

We synthesize a B_z and V_x time series at each of the five probes during a model NENL onset at X_NENL = -22 R_E, and demonstrate the multi-point signatures THEMIS would record.

Magnetotail reconnection의 정형적 시그너처는 X-line earthward에서 B_z > 0 (dipolarization), tailward에서 B_z < 0 (plasmoid), V_x는 두 방향으로 분기, 그리고 energetic 입자의 시간 분산이다. X_NENL = -22 R_E에서의 NENL onset 시나리오로 다섯 위성 각각의 B_z와 V_x 시계열을 합성한다.

In [ ]:
def synthesize_reconnection_timeseries(
    x_probe_re: float,
    x_xline_re: float,
    t_seconds: np.ndarray,
    onset_time_s: float,
    bbf_speed_km_s: float = 800.0,
    plasmoid_speed_km_s: float = 600.0,
    bz_amp_nt: float = 12.0,
    rng: np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Build synthetic B_z and V_x signals at a probe during reconnection.

    The reconnection X-line at x_xline_re launches an earthward BBF and a tailward
    plasmoid. The probe sees a B_z step (dipolarization or south-then-recovery)
    once the front arrives, with overshoot characteristic of bursty flows.

    Args:
        x_probe_re: Probe X position (R_E, negative).
        x_xline_re: X-line position (R_E, negative).
        t_seconds: Time array (s) starting before the onset.
        onset_time_s: Time of reconnection onset within ``t_seconds``.
        bbf_speed_km_s: Earthward BBF speed (km/s).
        plasmoid_speed_km_s: Tailward plasmoid speed (km/s).
        bz_amp_nt: Asymptotic B_z amplitude after dipolarization (nT).
        rng: Optional random generator for noise.

    Returns:
        Tuple (bz_nt, vx_km_s).
    """
    if rng is None:
        rng = np.random.default_rng()
    earthward = x_probe_re > x_xline_re
    distance_re = abs(x_probe_re - x_xline_re)
    distance_km = distance_re * R_E_KM
    speed = bbf_speed_km_s if earthward else plasmoid_speed_km_s
    arrival = onset_time_s + distance_km / speed
    # Sigmoid arrival profile.
    rise_tau = 8.0
    arg = (t_seconds - arrival) / rise_tau
    sigmoid = 1.0 / (1.0 + np.exp(-arg))
    sign = +1.0 if earthward else -1.0
    bz = sign * bz_amp_nt * sigmoid + 2.0 + 0.6 * rng.standard_normal(t_seconds.size)
    vx = sign * speed * sigmoid * np.exp(-((t_seconds - arrival - 30.0) / 90.0) ** 2) \
         + 30.0 * rng.standard_normal(t_seconds.size)
    return bz, vx


X_XLINE = -22.0  # NENL X-line at -22 R_E (within paper's 20-30 range)
ONSET = 60.0     # onset at t = 60 s in our synthetic frame
TIME = np.linspace(0.0, 240.0, 1201)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
ax_bz, ax_vx = axes

for probe in probe_order:
    xp = PROBE_X[probe]
    bz, vx = synthesize_reconnection_timeseries(xp, X_XLINE, TIME, ONSET, rng=RNG)
    color = next((p['color'] for k, p in PROBES.items() if k.startswith(probe)), 'k')
    ax_bz.plot(TIME, bz, color=color, lw=1.3, label=f'{probe} (X={xp:+.0f} R_E)')
    ax_vx.plot(TIME, vx, color=color, lw=1.3)

for ax in axes:
    ax.axvline(ONSET, color='k', lw=0.6, ls=':', label='reconnection onset')
    ax.axhline(0, color='k', lw=0.4)
ax_bz.set_ylabel('B_z (nT)')
ax_bz.set_title(f'Synthesized B_z at probes — NENL X-line at X = {X_XLINE} R_E')
ax_bz.legend(loc='upper right', fontsize=9, ncol=2)
ax_vx.set_ylabel('V_x (km/s)')
ax_vx.set_xlabel('Time (s)')
ax_vx.set_title('Synthesized V_x — earthward BBF (V_x > 0) vs tailward plasmoid (V_x < 0)')
plt.tight_layout()
plt.show()

### 3.1 Dispersion-Based X-Line Localization / 분산 기반 X-line 위치 추정

Given the arrival times τ_i at each probe X position x_i, we can least-squares-fit a single X-line position x_X and a wave speed v under the assumption τ_i = (x_i − x_X) / v. This is the simplest version of the timing analysis (Sect. 4.3 of the notes).

각 위성의 신호 도달 시간 τ_i와 위치 x_i가 주어지면 τ_i = (x_i − x_X) / v 가정 하에서 X-line 위치 x_X와 전파 속도 v를 최소제곱으로 추정할 수 있다.

In [ ]:
def detect_arrival(t: np.ndarray, signal: np.ndarray, threshold: float) -> float:
    """Find first time the absolute signal exceeds the threshold.

    Args:
        t: Time array (s).
        signal: Signal array.
        threshold: Detection threshold (same units as signal).

    Returns:
        Arrival time in seconds, or NaN if never crossed.
    """
    crossings = np.where(np.abs(signal) > threshold)[0]
    return t[crossings[0]] if crossings.size else np.nan


# Re-synthesize with quieter noise for arrival detection.
RNG_DETECT = np.random.default_rng(seed=11)
x_array = []
tau_array = []
for probe in probe_order:
    xp = PROBE_X[probe]
    bz, _ = synthesize_reconnection_timeseries(xp, X_XLINE, TIME, ONSET, rng=RNG_DETECT)
    arrival = detect_arrival(TIME, bz - bz[:int(ONSET * 5)].mean(), threshold=4.0)
    x_array.append(xp)
    tau_array.append(arrival - ONSET)
    print(f"  {probe}: x = {xp:+5.1f} R_E, τ = {arrival - ONSET:6.2f} s")

x_arr = np.asarray(x_array)
tau_arr = np.asarray(tau_array)

# τ = |x - x_X| / v. Solve for two branches.
# We assume earthward branch (x > x_X, v_BBF > 0): τ = (x - x_X)/v.
# Linear least squares: τ = (1/v) x - (x_X/v).
earthward_mask = x_arr > -22.0  # known geometry; in real data we test both branches
x_e, tau_e = x_arr[earthward_mask], tau_arr[earthward_mask]
A = np.vstack([x_e, np.ones_like(x_e)]).T
slope, intercept = np.linalg.lstsq(A, tau_e, rcond=None)[0]
v_fit = 1.0 / slope * R_E_KM   # km/s (slope has units s/R_E so 1/slope = R_E/s, * R_E_KM = km/s)
x_X_fit = -intercept / slope

print('\nFit using earthward-branch probes only:')
print(f'  Recovered X-line position: x_X = {x_X_fit:+.2f} R_E (true = {X_XLINE:+.1f})')
print(f'  Recovered propagation speed: v = {v_fit:+.0f} km/s (true = +800 km/s)')

## Part 4: GBO Conjugate Mapping / GBO 공역 사상

THEMIS's 20+ all-sky imagers (ASI) and ground magnetometers (GMAG) span ~12 hours of local time across northern Canada and Alaska. A magnetic-equator probe location is mapped to the ionosphere along magnetic field lines using a simple dipole approximation: a probe at equatorial distance L_eq R_E maps to invariant latitude Λ = arccos(1/√L_eq).

We map P5/P3/P4/P2/P1 apogee positions and overlay candidate GBO ASI footprints (Fort Yukon, Inuvik, Gillam, Sanikiluaq, etc.).

THEMIS의 ASI/GMAG 지상망은 캐나다 북부와 알래스카에 걸쳐 ~12시간의 local time을 커버한다. 단순 dipole 근사 하에서 자기 적도면의 probe 위치 L_eq R_E는 invariant latitude Λ = arccos(1/√L_eq)에 사상된다. 다섯 위성의 apogee 위치를 ionosphere로 사상하고 주요 GBO 사이트와 비교한다.

In [ ]:
def dipole_invariant_latitude(l_eq_re: float) -> float:
    """Map an equatorial L-shell to invariant latitude in a dipole field.

    Args:
        l_eq_re: Equatorial L-shell value in R_E.

    Returns:
        Invariant latitude in degrees.
    """
    return np.degrees(np.arccos(1.0 / np.sqrt(l_eq_re)))


def stretched_tail_latitude(x_re: float, b_lobe_nt: float = 25.0, b_polar_nt: float = 60000.0) -> float:
    """Approximate auroral-zone latitude for a stretched tail field line using flux conservation.

    The polar-cap flux beyond the open-closed boundary equals the lobe flux.
    For a footpoint at colatitude θ_c, polar-cap flux ≈ B_polar * 2π R_E^2 (1 - cos θ_c).
    Tail-lobe flux ≈ B_lobe * π R_T^2 with R_T ~ |X|/2 (paper-style estimate).

    Args:
        x_re: Probe X position in R_E (use absolute value).
        b_lobe_nt: Tail-lobe field strength in nT.
        b_polar_nt: Polar surface field in nT.

    Returns:
        Invariant latitude (deg).
    """
    r_t = abs(x_re) / 2.0
    flux_ratio = 0.5 * b_lobe_nt * r_t ** 2 / b_polar_nt
    cos_theta = max(min(1.0 - flux_ratio, 1.0), -1.0)
    return 90.0 - np.degrees(np.arccos(cos_theta))


# Mapping for each probe.
probe_mapping = []
for probe in probe_order:
    xp = abs(PROBE_X[probe])
    lat_dipole = dipole_invariant_latitude(xp)
    lat_stretched = stretched_tail_latitude(xp)
    probe_mapping.append((probe, xp, lat_dipole, lat_stretched))

print(f"{'Probe':<6} {'L_eq (R_E)':>12} {'Λ_dipole (°)':>15} {'Λ_stretched (°)':>17}")
print('-' * 55)
for p, xp, ld, ls in probe_mapping:
    print(f"{p:<6} {xp:12.1f} {ld:15.2f} {ls:17.2f}")

# THEMIS GBO ASI sites (representative subset, from Mende et al. 2008).
GBO_SITES = [
    ('Fort Yukon (FYKN)',  66.6, 214.8),
    ('Inuvik (INUV)',       68.4, 226.3),
    ('Whitehorse (WHIT)',   60.6, 224.8),
    ('Fort Smith (FSMI)',   60.0, 248.1),
    ('Gillam (GILL)',       56.4, 265.3),
    ('Rankin Inlet (RANK)', 62.8, 267.9),
    ('Sanikiluaq (SNKQ)',   56.5, 280.8),
    ('Kuujjuaq (KUUJ)',     58.1, 291.6),
]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: ground footprint map (longitude vs latitude).
ax_map = axes[0]
lats_dipole = np.array([m[2] for m in probe_mapping])
lats_stretched = np.array([m[3] for m in probe_mapping])
names = [m[0] for m in probe_mapping]
for site, lat, lon in GBO_SITES:
    ax_map.plot(lon, lat, 'k^', ms=10)
    ax_map.text(lon + 1.5, lat, site, fontsize=8, va='center')

# Probe footpoints projected onto magnetic midnight meridian (~285° MLong on 2008-02-02).
MIDNIGHT_LON = 285.0
for name, _, ld, ls in probe_mapping:
    color = next((p['color'] for k, p in PROBES.items() if k.startswith(name)), 'k')
    ax_map.plot(MIDNIGHT_LON, ls, 'o', ms=11, color=color, mec='k', mew=1)
    ax_map.text(MIDNIGHT_LON + 1.5, ls + 0.4, f'{name} ({ls:.1f}°)', fontsize=9, color=color, fontweight='bold')

ax_map.axvline(MIDNIGHT_LON, color='k', ls='--', lw=0.6, alpha=0.5, label='magnetic midnight')
ax_map.set_xlim(200, 310)
ax_map.set_ylim(54, 76)
ax_map.set_xlabel('Geographic longitude (°E)')
ax_map.set_ylabel('Geographic latitude (°N)')
ax_map.set_title('THEMIS GBO sites and probe footpoints (stretched-tail mapping)')
ax_map.legend(loc='lower right')

# Right: comparison of dipole vs stretched mapping.
ax_cmp = axes[1]
x_pos = np.arange(len(names))
ax_cmp.bar(x_pos - 0.2, lats_dipole, width=0.4, label='Pure dipole', color='steelblue', edgecolor='k')
ax_cmp.bar(x_pos + 0.2, lats_stretched, width=0.4, label='Stretched tail (flux conservation)', color='salmon', edgecolor='k')
ax_cmp.set_xticks(x_pos)
ax_cmp.set_xticklabels(names)
ax_cmp.set_ylabel('Invariant latitude (°)')
ax_cmp.set_title('Dipole vs stretched mapping — auroral-zone footprints')
ax_cmp.axhspan(64, 70, color='green', alpha=0.15, label='auroral oval (substorm)')
ax_cmp.legend(loc='lower left')
ax_cmp.set_ylim(50, 90)

plt.tight_layout()
plt.show()

### 4.1 Substorm Onset Localization on the Ground / 부폭풍 onset 위치 추정

We finally simulate an auroral break-up at a specific MLT/MLAT, generate ASI brightness time series from each GBO site (Gaussian beacon + noise), and determine the onset time and location by triangulating earliest-brightening pixels. This emulates the THEMIS-GBO inversion of magnetotail-to-ionosphere event mapping with δMLT ~ 6° and δXY ~ 1 R_E^2 requirements (Sect. 4.1 of the notes).

특정 MLT/MLAT에서 auroral break-up을 합성하고, GBO 사이트별 brightness 시계열을 만들어 가장 먼저 밝아진 픽셀을 삼각측량으로 onset 위치를 추정한다. THEMIS-GBO의 magnetotail-to-ionosphere 사상 역방향 추정을 흉내내며 δMLT ~ 6°, δXY ~ 1 R_E² 요구치를 검증한다.

In [ ]:
def asi_brightness(
    site_lat: float,
    site_lon: float,
    onset_lat: float,
    onset_lon: float,
    onset_time_s: float,
    t_seconds: np.ndarray,
    sigma_deg: float = 3.0,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """Synthesize ASI brightness curve at a GBO site for a localized onset.

    Brightness = exp(-distance²/2σ²) * sigmoid step at onset_time_s.

    Args:
        site_lat: Site latitude (deg).
        site_lon: Site longitude (deg).
        onset_lat: Onset arc latitude (deg).
        onset_lon: Onset arc longitude (deg).
        onset_time_s: Onset time within ``t_seconds`` (s).
        t_seconds: Time array (s).
        sigma_deg: Spatial decay scale of brightness (deg).
        rng: Optional random generator for shot noise.

    Returns:
        Brightness array (arbitrary kR units).
    """
    if rng is None:
        rng = np.random.default_rng()
    # Approximate degrees-to-degrees distance (high latitude, small region).
    dlat = site_lat - onset_lat
    dlon = (site_lon - onset_lon) * np.cos(np.deg2rad(onset_lat))
    distance_deg = np.hypot(dlat, dlon)
    spatial = np.exp(-0.5 * (distance_deg / sigma_deg) ** 2)
    sigmoid = 1.0 / (1.0 + np.exp(-(t_seconds - onset_time_s) / 4.0))
    return 8.0 * spatial * sigmoid + 0.4 + 0.15 * rng.standard_normal(t_seconds.size)


# Synthetic onset arc location (e.g., near Gillam, magnetic midnight 2008-02-26).
ONSET_LAT = 66.0
ONSET_LON = 268.0
ONSET_T = 30.0
TIME_GBO = np.linspace(0.0, 90.0, 181)  # 0.5 s cadence

RNG_GBO = np.random.default_rng(seed=42)

brightness_curves = {}
for site_name, lat, lon in GBO_SITES:
    brightness_curves[site_name] = asi_brightness(
        lat, lon, ONSET_LAT, ONSET_LON, ONSET_T, TIME_GBO, rng=RNG_GBO
    )

# Detect first significant brightening per site.
site_arrivals = {}
baseline = 0.6
threshold = 1.5
for site_name, curve in brightness_curves.items():
    above = np.where(curve > threshold)[0]
    site_arrivals[site_name] = TIME_GBO[above[0]] if above.size else np.nan

# Plot brightness stack and onset estimate.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax_curves, ax_loc = axes

for (site_name, curve), (_, lat, lon) in zip(brightness_curves.items(), GBO_SITES):
    ax_curves.plot(TIME_GBO, curve, lw=1.2, label=f"{site_name} ({lat:.1f}°,{lon:.1f}°)")
ax_curves.axvline(ONSET_T, color='k', ls='--', lw=0.8, label='true onset')
ax_curves.set_xlabel('Time (s)')
ax_curves.set_ylabel('ASI brightness (kR)')
ax_curves.set_title('Synthetic ASI brightness time series at GBO sites')
ax_curves.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.0, 1.0))

# Locate onset by inverse-distance weighting of arrival times: earliest = nearest.
lats_arr = np.array([s[1] for s in GBO_SITES])
lons_arr = np.array([s[2] for s in GBO_SITES])
arrivals = np.array([site_arrivals[s[0]] for s in GBO_SITES])
weights = 1.0 / np.maximum(arrivals - ONSET_T + 0.5, 0.1) ** 2
lat_est = np.sum(weights * lats_arr) / np.sum(weights)
lon_est = np.sum(weights * lons_arr) / np.sum(weights)

ax_loc.scatter(lons_arr, lats_arr, c=arrivals, cmap='viridis_r', s=140, edgecolor='k', zorder=4)
for (site_name, lat, lon), tarr in zip(GBO_SITES, arrivals):
    ax_loc.text(lon + 1, lat + 0.2, f'{site_name.split()[0]}\n{tarr:.1f}s', fontsize=7)
ax_loc.plot(ONSET_LON, ONSET_LAT, 'r*', ms=22, label=f'true onset ({ONSET_LAT:.1f}°,{ONSET_LON:.1f}°)')
ax_loc.plot(lon_est, lat_est, 'b+', ms=22, mew=3, label=f'estimated ({lat_est:.1f}°,{lon_est:.1f}°)')
ax_loc.set_xlabel('Geographic longitude (°E)')
ax_loc.set_ylabel('Geographic latitude (°N)')
ax_loc.set_title('Onset localization from earliest-brightening triangulation')
ax_loc.legend(loc='lower left')
ax_loc.set_xlim(205, 305)
ax_loc.set_ylim(53, 72)

cbar = plt.colorbar(ax_loc.collections[0], ax=ax_loc)
cbar.set_label('Detection time (s)')
plt.tight_layout()
plt.show()

loc_err_lat = lat_est - ONSET_LAT
loc_err_lon = (lon_est - ONSET_LON) * np.cos(np.deg2rad(ONSET_LAT))
loc_err_deg = np.hypot(loc_err_lat, loc_err_lon)
print(f'\nOnset localization error: {loc_err_deg:.2f} ° (~{loc_err_deg * 111:.0f} km)')
print(f'Paper requirement: δMLT ≲ 6° (~600 km), δXY ≲ 1 R_E² at the inner plasma-sheet edge.')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Implementation Here / 본 노트북 구현 |
|---|---|---|
| Apogee-aligned 5-probe constellation / 5위성 apogee 정렬 | Sect. 3.1 — periods 4:2:1:1:7/8 sidereal days | Part 1: Kepler orbits + radial-distance recurrence |
| NENL vs CD timing / NENL과 CD 시간 비교 | Tables 2 & 3 — 30 s vs 120 s break-up offset | Part 2: arrival-time stack plots and decision rule |
| Multi-point reconnection signatures / 다지점 reconnection | B_z reversal, BBF (+800 km/s) vs plasmoid (-600 km/s) | Part 3: synthetic B_z/V_x and least-squares X-line fit |
| GBO conjugate mapping / 공역 사상 | δMLT ≲ 6°, δXY ≲ 1 R_E² required | Part 4: dipole/stretched mapping + ASI triangulation |

### Key Insights / 핵심 통찰

1. **Integer period ratios are the design trick.** With T_P1:T_P2:T_P3,P4:T_P5 = 4:2:1:1:(7/8) sidereal days, the apogee-on-midnight conjunction recurs deterministically — not by chance. This trick converts a one-off conjunction into a calendar of guaranteed events.

   **정수배 주기 비가 설계의 핵심이다.** 4:2:1:1:7/8 sidereal days라는 단순한 비율이 자정 자오선 conjunction을 예측 가능한 캘린더로 바꾸어 놓는다.

2. **The decisive observable is the inner-vs-outer arrival-time order.** If the disturbance reaches P5 (~10 R_E) before P1 (~30 R_E), CD wins; reversed, NENL wins. THEMIS in 2008 announced NENL preceding break-up by ~96 s — a clean NENL signature.

   **결정적 관측량은 내·외측 위성의 도달 시간 순서이다.** 신호가 P5 → P1 순으로 도달하면 CD, 반대 순서면 NENL이며, 2008년 THEMIS는 NENL이 break-up에 ~96 s 선행함을 보고했다.

3. **GBO is not optional.** The auroral break-up that defines the substorm onset is intrinsically optical, so without ground ASIs the in-situ probes cannot anchor the t = 0. The 12-hour-of-MLT GBO chain provides this anchor.

   **GBO는 선택이 아니다.** 부폭풍 onset의 정의(auroral break-up)는 본질적으로 ionospheric/optical 현상이므로, 12시간의 MLT를 커버하는 ASI 네트워크가 t = 0의 ground truth를 제공한다.

4. **The mapping ambiguity is non-trivial.** Pure-dipole mapping places L = 10 R_E at Λ = 71.6°, but the magnetotail is stretched, putting the true footprint near Λ = 65–67° (auroral oval). Reconciling probe-to-ground requires an empirical magnetic field model (T96, T01) at runtime.

   **사상 모호성은 자명하지 않다.** Pure dipole에서는 L = 10 R_E가 Λ = 71.6°지만, 늘어난 magnetotail에서는 실제 footprint가 Λ ~ 65–67°(auroral oval)에 있다. 운영 시에는 T96/T01 같은 경험적 자기장 모델이 필요하다.

### References / 참고문헌

- Angelopoulos, V. "The THEMIS Mission", *Space Sci. Rev.* **141**, 5–34, 2008. DOI 10.1007/s11214-008-9336-1
- Angelopoulos, V., et al. "Tail reconnection triggering substorm onset", *Science* **321**, 931–935, 2008.
- Mende, S. B., et al. "The THEMIS Array of Ground-Based Observatories", *Space Sci. Rev.* (2008).
- McPherron, R. L., Russell, C. T., Aubry, M. P. "Satellite studies of magnetospheric substorms on August 15, 1968", *J. Geophys. Res.* **78**, 3131, 1973.
